# Trust Region Q Adjoint Matching Executable Report

This notebook is the control surface for running TRQAM directly in the Jupyter kernel and rendering the resulting CSV logs into a compact report.

The default profile is now `full_bc`: running all cells starts the full 300K-step BC pretraining job for `humanoidmaze-medium`. It follows the paper's humanoidmaze/common hyperparameters, saves only the final BC checkpoint, and skips in-training eval for speed. After BC finishes, switch `RUN_PROFILE` to `full_offline_trqam` or `full_offline_qam` for offline RL; those RL profiles evaluate SR every 1K steps and keep only the latest checkpoint snapshot.


In [ ]:
# Report display controls. Keep REPORT_MODE=False while working in Jupyter.
REPORT_MODE = False

from IPython.display import HTML, display

REPORT_CSS = r'''
<style>
:root {
  --trqam-text: #1f2328;
  --trqam-muted: #57606a;
  --trqam-border: #d0d7de;
  --trqam-accent: #0969da;
  --trqam-bg: #ffffff;
  --trqam-soft: #f6f8fa;
}
body, .jp-Notebook, .notebook_app {
  color: var(--trqam-text);
}
.trqam-callout {
  border-left: 4px solid var(--trqam-accent);
  background: var(--trqam-soft);
  padding: 12px 14px;
  margin: 10px 0 16px;
}
.trqam-kpi-grid {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(150px, 1fr));
  gap: 10px;
  margin: 12px 0 18px;
}
.trqam-kpi {
  border: 1px solid var(--trqam-border);
  border-radius: 6px;
  padding: 10px 12px;
  background: var(--trqam-bg);
}
.trqam-kpi .label {
  color: var(--trqam-muted);
  font-size: 12px;
  margin-bottom: 4px;
}
.trqam-kpi .value {
  font-weight: 650;
  font-size: 18px;
}
.trqam-report h2, .trqam-report h3 {
  margin-top: 18px;
}
.trqam-report table, table.dataframe, .report-table {
  border-collapse: collapse;
  width: 100%;
  font-size: 13px;
}
.trqam-report th, .trqam-report td, table.dataframe th, table.dataframe td, .report-table th, .report-table td {
  border: 1px solid var(--trqam-border);
  padding: 6px 8px;
  text-align: left;
  vertical-align: top;
}
.trqam-report th, table.dataframe th, .report-table th {
  background: var(--trqam-soft);
}
.trqam-muted { color: var(--trqam-muted); }
@media print {
  .jp-Toolbar, .jp-NotebookPanel-toolbar, .jp-Cell-inputWrapper, .jp-InputPrompt, .jp-OutputPrompt, div.input, .prompt {
    display: none !important;
  }
}
</style>
'''

HIDE_CODE_CSS = r'''
<style id="trqam-hide-code">
.jp-Cell-inputWrapper, .jp-InputPrompt, .jp-OutputPrompt, div.input, .prompt {
  display: none !important;
}
</style>
'''

SHOW_CODE_CSS = r'''
<style id="trqam-show-code">
.jp-Cell-inputWrapper, .jp-InputPrompt, .jp-OutputPrompt, div.input, .prompt {
  display: revert !important;
}
.jp-Cell-inputWrapper {
  display: flex !important;
}
</style>
<script>
document.getElementById('trqam-hide-code')?.remove();
</script>
'''

if REPORT_MODE:
    display(HTML(REPORT_CSS + HIDE_CODE_CSS))
else:
    display(HTML(REPORT_CSS + SHOW_CODE_CSS))


## 1. Background

TRQAM fine-tunes a pretrained flow policy with off-policy RL while constraining the updated policy to stay close to the base policy under a path-space KL trust region. The Q-function action gradient is used as an adjoint signal for improving the policy vector field. When the estimated KL exceeds the configured budget, dual descent increases the trust-region coefficient and makes the policy update more conservative.

The paper pipeline has two stages.

1. Train a flow policy with behavior cloning by setting `bc_only=True`.
2. Load the BC checkpoint as `pretrained_actor_path` and fine-tune with TRQAM.


In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

main_figure = Path('assets/Main_figure.png')
if main_figure.exists():
    display(Image(filename=str(main_figure)))
else:
    display(Markdown('_Main figure image was not found at `assets/Main_figure.png`._'))


## 2. Runtime Assumptions

Run this notebook from the repository root with the `TRQAM Conda (GPU)` kernel. The notebook sets `MUJOCO_GL=egl`, `WANDB_MODE=disabled`, `XLA_PYTHON_CLIENT_PREALLOCATE=false`, and defaults `CUDA_VISIBLE_DEVICES=1` before importing JAX.

If you want a different GPU, set `CUDA_VISIBLE_DEVICES` before starting the kernel, or edit the runtime setup cell and restart the kernel before running the notebook. JAX reads GPU visibility at import time, so changing it after `import jax` will not reliably move the run.


In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '1')

ROOT = Path.cwd()
if not (ROOT / 'main.py').exists() and (ROOT / 'trqam' / 'main.py').exists():
    ROOT = ROOT / 'trqam'

expected_python = '/home/sanghyeok/miniconda3/envs/trqam/bin/python'
expected_prefix = '/home/sanghyeok/miniconda3/envs/trqam'
current_python = sys.executable
current_prefix = sys.prefix

required_packages = ['jax', 'flax', 'distrax', 'pandas', 'matplotlib', 'tqdm', 'ml_collections', 'ogbench']
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]
if missing_packages:
    raise RuntimeError(
        'Missing packages: ' + ', '.join(missing_packages) + '\n'
        f'Current Python: {current_python}\n'
        f'Current prefix: {current_prefix}\n'
        f'Expected Python: {expected_python}\n'
        f'Expected prefix: {expected_prefix}\n'
        'Select the Jupyter kernel named Python (trqam), then restart the kernel and run from the top.'
    )

import glob
import html
import json
import pickle
import random
import re
import shutil
import subprocess
import time
from collections import defaultdict
from dataclasses import asdict, dataclass
from typing import Optional

import flax
import jax
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display
from tqdm.auto import trange

ROOT = Path.cwd()
if not (ROOT / 'main.py').exists() and (ROOT / 'trqam' / 'main.py').exists():
    ROOT = ROOT / 'trqam'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agents import agents
from agents.qam import get_config as get_qam_config
from agents.trqam import get_config as get_trqam_config
from envs.env_utils import make_env_and_datasets
from envs.ogbench_utils import make_ogbench_env_and_datasets
from evaluation import evaluate
from utils.datasets import Dataset, ReplayBuffer
from utils.flax_utils import restore_agent_with_file, save_agent


def is_robomimic_env(env_name):
    if 'low_dim' not in env_name:
        return False
    task, dataset_type, _ = env_name.split('-')
    return task in ('lift', 'can', 'square', 'transport', 'tool_hang') and dataset_type in ('mh', 'ph')


def display_kv(title, rows):
    df = pd.DataFrame(rows, columns=['Item', 'Value'])
    display(Markdown(f'### {title}'))
    display(df)
    return df


def fmt_value(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (tuple, list)):
        return ', '.join(map(str, value))
    return str(value)

runtime_info = {
    'workspace': str(ROOT),
    'jax_backend': jax.default_backend(),
    'jax_devices': ', '.join(str(device) for device in jax.devices()),
    'wandb_mode': os.environ.get('WANDB_MODE', ''),
    'mujoco_gl': os.environ.get('MUJOCO_GL', ''),
    'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES', ''),
}

display_kv(
    'Runtime Check',
    [
        ('Workspace', f'`{runtime_info["workspace"]}`'),
        ('JAX backend', f'`{runtime_info["jax_backend"]}`'),
        ('JAX devices', f'`{runtime_info["jax_devices"]}`'),
        ('WANDB_MODE', f'`{runtime_info["wandb_mode"]}`'),
        ('MUJOCO_GL', f'`{runtime_info["mujoco_gl"]}`'),
        ('CUDA_VISIBLE_DEVICES', f'`{runtime_info["cuda_visible_devices"]}`'),
    ],
)

if runtime_info['jax_backend'] != 'gpu':
    display(Markdown('> **Warning:** this notebook kernel does not currently see a JAX GPU backend. Start the Jupyter kernel inside a GPU allocation for GPU execution.'))


## 3. Experiment Settings

`NotebookConfig` mirrors the important `main.py` flags in a form that is easy to edit inside the notebook. The default mode is notebook-local full BC pretraining.

- `RUN_PROFILE='full_bc'`: full 300K-step humanoidmaze-medium BC pretraining in this notebook kernel
- `RUN_PROFILE='full_offline_trqam'`: full humanoidmaze-medium TRQAM offline RL fine-tuning from a BC checkpoint
- `RUN_PROFILE='full_offline_qam'`: full humanoidmaze-medium QAM offline RL fine-tuning from the same BC checkpoint
- `RUN_PROFILE='cube_triple_bc'`: full 300K-step cube-triple BC pretraining
- `RUN_PROFILE='cube_triple_offline_trqam'`: full cube-triple TRQAM offline RL fine-tuning
- `RUN_PROFILE='cube_triple_offline_qam'`: full cube-triple QAM offline RL fine-tuning
- `RUN_PROFILE='smoke'`: short notebook-local sanity check

For `humanoidmaze-*`, the notebook applies the paper settings: `discount=0.999`, `rho=0.0`, action chunk horizon 1, batch size 256, learning rate 3e-4, flow steps 10, width 512, depth 4, BC 300K steps, offline RL 1M steps, and TRQAM `kl_budget=0.5`. For `cube-triple`, it uses action chunk horizon 1, width 1024, `actor_layer_norm=True`, and default critic discount/rho; it uses the 10M npz directory when present and otherwise falls back to cached OGBench data. By default it uses physical GPU 1 via `CUDA_VISIBLE_DEVICES=1`; edit the runtime setup cell or launch the kernel with a different value to choose another GPU.


In [ ]:
@dataclass
class NotebookConfig:
    agent_name: str = 'trqam'
    run_group: str = 'notebook_report'
    tags: str = 'TRQAM_NOTEBOOK'
    seed: int = 10001
    env_name: str = 'humanoidmaze-medium-navigate-singletask-task1-v0'
    save_dir: str = 'exp_full'
    ogbench_dataset_dir: Optional[str] = None
    pretrained_actor_path: Optional[str] = None

    offline_steps: int = 1
    online_steps: int = 0
    log_interval: int = 1
    eval_interval: int = 1
    save_interval: int = 0
    keep_last_checkpoint_only: bool = False
    async_eval: bool = False
    async_eval_cuda_visible_devices: Optional[str] = None
    eval_episodes: int = 1
    video_episodes: int = 0
    video_frame_skip: int = 3

    horizon_length: int = 1
    sparse: bool = False
    action_chunking: bool = True
    bc_only: bool = True
    kl_budget: float = 0.5
    kl_budget_online: Optional[float] = None
    inv_temp: float = 3.0
    fql_alpha: float = 0.0
    edit_scale: float = 0.0

    dataset_proportion: float = 0.001
    dataset_replace_interval: int = 0
    buffer_size: int = 1_000_000
    start_training: int = 5
    utd_ratio: int = 1
    balanced_sampling: bool = False
    large_network: bool = False

    batch_size: int = 8
    actor_hidden_dims: tuple = (64, 64)
    value_hidden_dims: tuple = (64, 64)
    actor_layer_norm: bool = False
    flow_steps: int = 2


# Main notebook controls.
RUN_PROFILE = os.environ.get('TRQAM_NOTEBOOK_PROFILE', 'full_bc')
EXECUTION_MODE = 'notebook'
PRETRAINED_ACTOR_PATH = os.environ.get('TRQAM_PRETRAINED_ACTOR_PATH') or None
OGBENCH_DATASET_DIR = os.environ.get('TRQAM_OGBENCH_DATASET_DIR') or None

FULL_WIDTH = int(os.environ.get('TRQAM_FULL_WIDTH', '512'))
FULL_HIDDEN_DIMS = (FULL_WIDTH, FULL_WIDTH, FULL_WIDTH, FULL_WIDTH)
FULL_ACTOR_LAYER_NORM = os.environ.get('TRQAM_ACTOR_LAYER_NORM', 'False').lower() in {'1', 'true', 'yes'}
FULL_DATASET_REPLACE_INTERVAL = int(os.environ.get('TRQAM_DATASET_REPLACE_INTERVAL', '1000' if OGBENCH_DATASET_DIR else '0'))
CUBE_TRIPLE_ENV = 'cube-triple-play-singletask-task2-v0'
_CUBE_TRIPLE_10M_DIR = Path.home() / '.ogbench' / 'data' / 'cube-triple-play-10m-v0'
_CUBE_TRIPLE_DATASET_DIR_OVERRIDE = os.environ.get('TRQAM_CUBE_TRIPLE_DATASET_DIR')
CUBE_TRIPLE_DATASET_DIR = _CUBE_TRIPLE_DATASET_DIR_OVERRIDE or (str(_CUBE_TRIPLE_10M_DIR) if _CUBE_TRIPLE_10M_DIR.exists() else None)
CUBE_TRIPLE_DATASET_REPLACE_INTERVAL = int(os.environ.get('TRQAM_CUBE_TRIPLE_DATASET_REPLACE_INTERVAL', '1000' if CUBE_TRIPLE_DATASET_DIR else '0'))
CUBE_WIDTH = int(os.environ.get('TRQAM_CUBE_WIDTH', '1024'))
CUBE_HIDDEN_DIMS = (CUBE_WIDTH, CUBE_WIDTH, CUBE_WIDTH, CUBE_WIDTH)
CUBE_ACTOR_LAYER_NORM = os.environ.get('TRQAM_CUBE_ACTOR_LAYER_NORM', 'True').lower() in {'1', 'true', 'yes'}
CUBE_ACTION_CHUNK_HORIZON = int(os.environ.get('TRQAM_CUBE_ACTION_CHUNK_HORIZON', '1'))
ASYNC_EVAL_CUDA_VISIBLE_DEVICES = os.environ.get('TRQAM_EVAL_CUDA_VISIBLE_DEVICES', '2')
SKIP_COMPLETED_RUNS = os.environ.get('TRQAM_SKIP_COMPLETED_RUNS', 'True').lower() in {'1', 'true', 'yes'}

PROFILE_OVERRIDES = {
    'smoke': dict(
        run_group='notebook_humanoid_medium_smoke',
        tags='HUMANOID_MEDIUM_NOTEBOOK_SMOKE',
        env_name='humanoidmaze-medium-navigate-singletask-task1-v0',
        seed=0,
        save_dir='exp_notebook',
        offline_steps=1,
        online_steps=0,
        log_interval=1,
        eval_interval=1,
        save_interval=0,
        eval_episodes=1,
        dataset_proportion=0.001,
        dataset_replace_interval=0,
        start_training=5,
        bc_only=True,
        batch_size=8,
        actor_hidden_dims=(64, 64),
        value_hidden_dims=(64, 64),
        actor_layer_norm=False,
        flow_steps=2,
    ),
    'full_bc': dict(
        run_group='bc_pretrain_humanoid_medium',
        tags='HUMANOID_MEDIUM_BC',
        env_name='humanoidmaze-medium-navigate-singletask-task1-v0',
        seed=10001,
        save_dir='exp_full',
        ogbench_dataset_dir=OGBENCH_DATASET_DIR,
        offline_steps=300_000,
        online_steps=0,
        log_interval=1_000,
        eval_interval=0,
        save_interval=300_000,
        keep_last_checkpoint_only=True,
        eval_episodes=50,
        dataset_proportion=1.0,
        dataset_replace_interval=FULL_DATASET_REPLACE_INTERVAL,
        start_training=5_000,
        bc_only=True,
        batch_size=256,
        actor_hidden_dims=FULL_HIDDEN_DIMS,
        value_hidden_dims=FULL_HIDDEN_DIMS,
        actor_layer_norm=FULL_ACTOR_LAYER_NORM,
        flow_steps=10,
    ),
    'full_offline_trqam': dict(
        agent_name='trqam',
        run_group='offline_rl_humanoid_medium',
        tags='HUMANOID_MEDIUM_OFFLINE_TRQAM',
        env_name='humanoidmaze-medium-navigate-singletask-task1-v0',
        seed=10001,
        save_dir='exp_full',
        ogbench_dataset_dir=OGBENCH_DATASET_DIR,
        pretrained_actor_path=PRETRAINED_ACTOR_PATH,
        offline_steps=1_000_000,
        online_steps=0,
        log_interval=1_000,
        eval_interval=1_000,
        save_interval=1_000,
        keep_last_checkpoint_only=True,
        async_eval=True,
        async_eval_cuda_visible_devices=ASYNC_EVAL_CUDA_VISIBLE_DEVICES,
        eval_episodes=50,
        dataset_proportion=1.0,
        dataset_replace_interval=FULL_DATASET_REPLACE_INTERVAL,
        start_training=5_000,
        bc_only=False,
        kl_budget=0.5,
        batch_size=256,
        actor_hidden_dims=FULL_HIDDEN_DIMS,
        value_hidden_dims=FULL_HIDDEN_DIMS,
        actor_layer_norm=FULL_ACTOR_LAYER_NORM,
        flow_steps=10,
    ),
    'full_offline_qam': dict(
        agent_name='qam',
        run_group='offline_qam_humanoid_medium',
        tags='HUMANOID_MEDIUM_QAM',
        env_name='humanoidmaze-medium-navigate-singletask-task1-v0',
        seed=10001,
        save_dir='exp_full',
        ogbench_dataset_dir=OGBENCH_DATASET_DIR,
        pretrained_actor_path=PRETRAINED_ACTOR_PATH,
        offline_steps=1_000_000,
        online_steps=0,
        log_interval=1_000,
        eval_interval=1_000,
        save_interval=1_000,
        keep_last_checkpoint_only=True,
        async_eval=True,
        async_eval_cuda_visible_devices=ASYNC_EVAL_CUDA_VISIBLE_DEVICES,
        eval_episodes=50,
        dataset_proportion=1.0,
        dataset_replace_interval=FULL_DATASET_REPLACE_INTERVAL,
        start_training=5_000,
        bc_only=False,
        batch_size=256,
        actor_hidden_dims=FULL_HIDDEN_DIMS,
        value_hidden_dims=FULL_HIDDEN_DIMS,
        actor_layer_norm=FULL_ACTOR_LAYER_NORM,
        flow_steps=10,
        inv_temp=3.0,
        fql_alpha=0.0,
        edit_scale=0.0,
    ),
    'cube_triple_bc': dict(
        run_group='bc_pretrain_cube_triple',
        tags='CUBE_TRIPLE_BC',
        env_name=CUBE_TRIPLE_ENV,
        seed=10001,
        save_dir='exp_full',
        ogbench_dataset_dir=CUBE_TRIPLE_DATASET_DIR,
        offline_steps=300_000,
        online_steps=0,
        log_interval=1_000,
        eval_interval=0,
        save_interval=300_000,
        keep_last_checkpoint_only=True,
        eval_episodes=50,
        dataset_proportion=1.0,
        dataset_replace_interval=CUBE_TRIPLE_DATASET_REPLACE_INTERVAL,
        start_training=5_000,
        horizon_length=CUBE_ACTION_CHUNK_HORIZON,
        bc_only=True,
        batch_size=256,
        actor_hidden_dims=CUBE_HIDDEN_DIMS,
        value_hidden_dims=CUBE_HIDDEN_DIMS,
        actor_layer_norm=CUBE_ACTOR_LAYER_NORM,
        flow_steps=10,
    ),
    'cube_triple_offline_trqam': dict(
        agent_name='trqam',
        run_group='offline_rl_cube_triple',
        tags='CUBE_TRIPLE_OFFLINE_TRQAM',
        env_name=CUBE_TRIPLE_ENV,
        seed=10001,
        save_dir='exp_full',
        ogbench_dataset_dir=CUBE_TRIPLE_DATASET_DIR,
        pretrained_actor_path=PRETRAINED_ACTOR_PATH,
        offline_steps=1_000_000,
        online_steps=0,
        log_interval=1_000,
        eval_interval=1_000,
        save_interval=1_000,
        keep_last_checkpoint_only=True,
        async_eval=True,
        async_eval_cuda_visible_devices=ASYNC_EVAL_CUDA_VISIBLE_DEVICES,
        eval_episodes=50,
        dataset_proportion=1.0,
        dataset_replace_interval=CUBE_TRIPLE_DATASET_REPLACE_INTERVAL,
        start_training=5_000,
        horizon_length=CUBE_ACTION_CHUNK_HORIZON,
        bc_only=False,
        kl_budget=0.5,
        batch_size=256,
        actor_hidden_dims=CUBE_HIDDEN_DIMS,
        value_hidden_dims=CUBE_HIDDEN_DIMS,
        actor_layer_norm=CUBE_ACTOR_LAYER_NORM,
        flow_steps=10,
    ),
    'cube_triple_offline_qam': dict(
        agent_name='qam',
        run_group='offline_qam_cube_triple',
        tags='CUBE_TRIPLE_QAM',
        env_name=CUBE_TRIPLE_ENV,
        seed=10001,
        save_dir='exp_full',
        ogbench_dataset_dir=CUBE_TRIPLE_DATASET_DIR,
        pretrained_actor_path=PRETRAINED_ACTOR_PATH,
        offline_steps=1_000_000,
        online_steps=0,
        log_interval=1_000,
        eval_interval=1_000,
        save_interval=1_000,
        keep_last_checkpoint_only=True,
        async_eval=True,
        async_eval_cuda_visible_devices=ASYNC_EVAL_CUDA_VISIBLE_DEVICES,
        eval_episodes=50,
        dataset_proportion=1.0,
        dataset_replace_interval=CUBE_TRIPLE_DATASET_REPLACE_INTERVAL,
        start_training=5_000,
        horizon_length=CUBE_ACTION_CHUNK_HORIZON,
        bc_only=False,
        batch_size=256,
        actor_hidden_dims=CUBE_HIDDEN_DIMS,
        value_hidden_dims=CUBE_HIDDEN_DIMS,
        actor_layer_norm=CUBE_ACTOR_LAYER_NORM,
        flow_steps=10,
        inv_temp=3.0,
        fql_alpha=0.0,
        edit_scale=0.0,
    ),
}

def find_latest_bc_checkpoint(save_dir, env_name):
    patterns = [
        Path(save_dir).glob(f'bc_pretrain_humanoid_medium/{env_name}/*/params_*.pkl'),
        Path(save_dir).glob(f'bc_pretrain_cube_triple/{env_name}/*/params_*.pkl'),
        Path(save_dir).glob(f'bc_pretrain*/{env_name}/*/params_*.pkl'),
        Path(save_dir).glob(f'bc_pretrain/{env_name}/*/params_*.pkl'),
        Path(save_dir).glob(f'*/bc_pretrain_humanoid_medium/{env_name}/*/params_*.pkl'),
        Path(save_dir).glob(f'*/bc_pretrain_cube_triple/{env_name}/*/params_*.pkl'),
        Path(save_dir).glob(f'*/bc_pretrain*/{env_name}/*/params_*.pkl'),
        Path(save_dir).glob(f'*/bc_pretrain/{env_name}/*/params_*.pkl'),
    ]
    candidates = [path for pattern in patterns for path in pattern]
    return str(max(candidates, key=lambda p: p.stat().st_mtime)) if candidates else None

RL_PROFILES = {'full_offline_trqam', 'full_offline_qam', 'cube_triple_offline_trqam', 'cube_triple_offline_qam'}

def configure_profile(profile):
    if profile not in PROFILE_OVERRIDES:
        raise ValueError(f'Unknown RUN_PROFILE={profile!r}. Choose one of {sorted(PROFILE_OVERRIDES)}')

    new_cfg = NotebookConfig(**PROFILE_OVERRIDES[profile])
    if profile in RL_PROFILES and new_cfg.pretrained_actor_path is None:
        new_cfg.pretrained_actor_path = find_latest_bc_checkpoint(new_cfg.save_dir, new_cfg.env_name)

    if profile in RL_PROFILES and not new_cfg.pretrained_actor_path:
        raise RuntimeError(
            f'{profile} notebook execution requires a BC checkpoint. '
            'Run full_bc first or set TRQAM_PRETRAINED_ACTOR_PATH=/path/to/params_300000.pkl.'
        )
    return new_cfg


cfg = configure_profile(RUN_PROFILE)

config_rows = [
    ('RUN_PROFILE', RUN_PROFILE),
    ('EXECUTION_MODE', EXECUTION_MODE),
    ('SKIP_COMPLETED_RUNS', SKIP_COMPLETED_RUNS),
] + [(key, fmt_value(value)) for key, value in asdict(cfg).items()]
display_kv('Experiment Configuration', config_rows)


In [ ]:
def build_agent_config(cfg):
    if cfg.agent_name == 'trqam':
        config = get_trqam_config()
    elif cfg.agent_name == 'qam':
        config = get_qam_config()
    else:
        raise ValueError(f'Unsupported agent_name={cfg.agent_name!r}')
    config['horizon_length'] = cfg.horizon_length
    config['action_chunking'] = cfg.action_chunking
    config['bc_only'] = cfg.bc_only
    if cfg.agent_name == 'trqam':
        config['kl_budget'] = cfg.kl_budget
    config['batch_size'] = cfg.batch_size
    config['actor_hidden_dims'] = tuple(cfg.actor_hidden_dims)
    config['value_hidden_dims'] = tuple(cfg.value_hidden_dims)
    config['actor_layer_norm'] = cfg.actor_layer_norm
    config['flow_steps'] = cfg.flow_steps
    if cfg.agent_name == 'qam':
        config['inv_temp'] = cfg.inv_temp
        config['fql_alpha'] = cfg.fql_alpha
        config['edit_scale'] = cfg.edit_scale

    if cfg.large_network:
        config['actor_hidden_dims'] = (1024, 1024, 1024, 1024)
        config['value_hidden_dims'] = (1024, 1024, 1024, 1024)
        config['actor_layer_norm'] = True

    if cfg.env_name.startswith('humanoidmaze-'):
        config['discount'] = 0.999
        config['rho'] = 0.0

    return config


agent_config = build_agent_config(cfg)
agent_config_rows = [
    ('agent_name', agent_config['agent_name']),
    ('batch_size', agent_config['batch_size']),
    ('actor_hidden_dims', fmt_value(agent_config['actor_hidden_dims'])),
    ('value_hidden_dims', fmt_value(agent_config['value_hidden_dims'])),
    ('actor_layer_norm', agent_config['actor_layer_norm']),
    ('flow_steps', agent_config['flow_steps']),
    ('horizon_length', agent_config['horizon_length']),
    ('action_chunking', agent_config['action_chunking']),
    ('bc_only', agent_config['bc_only']),
    ('kl_budget', agent_config.get('kl_budget', 'n/a')),
    ('inv_temp', agent_config.get('inv_temp', 'n/a')),
    ('fql_alpha', agent_config.get('fql_alpha', 'n/a')),
    ('edit_scale', agent_config.get('edit_scale', 'n/a')),
    ('discount', agent_config['discount']),
    ('rho', agent_config.get('rho')),
]
display_kv('Agent Configuration', agent_config_rows)


## 4. Dataset and Environment Loading

The notebook follows the same dataset path as `main.py`. For the default `humanoidmaze-medium-navigate-singletask-task1-v0`, OGBench loads the cached `humanoidmaze-medium-navigate-v0` dataset and relabels rewards/masks for the selected single-task goal.


In [ ]:
def load_env_and_datasets_for_notebook(cfg):
    dataset_paths = []

    if cfg.ogbench_dataset_dir is not None:
        dataset_paths = [
            path for path in sorted(glob.glob(str(Path(cfg.ogbench_dataset_dir).expanduser() / '*.npz')))
            if '-val.npz' not in path
        ]
        if cfg.dataset_proportion < 1.0:
            subset_size = max(1, int(len(dataset_paths) * cfg.dataset_proportion))
            dataset_paths = dataset_paths[:subset_size]
        if not dataset_paths:
            raise FileNotFoundError(f'No training npz files found in {cfg.ogbench_dataset_dir}')

        env, eval_env, train_dataset, val_dataset = make_ogbench_env_and_datasets(
            cfg.env_name,
            dataset_path=dataset_paths[0],
            compact_dataset=False,
        )
    else:
        env, eval_env, train_dataset, val_dataset = make_env_and_datasets(cfg.env_name)

    return env, eval_env, train_dataset, val_dataset, dataset_paths


def process_train_dataset(ds, cfg):
    ds = Dataset.create(**ds)

    if cfg.dataset_proportion < 1.0 and cfg.ogbench_dataset_dir is None:
        new_size = int(len(ds['masks']) * cfg.dataset_proportion)
        ds = Dataset.create(**{key: value[:new_size] for key, value in ds.items()})

    if is_robomimic_env(cfg.env_name):
        ds_dict = {key: value for key, value in ds.items()}
        ds_dict['rewards'] = ds['rewards'] - 1.0
        ds = Dataset.create(**ds_dict)

    if cfg.sparse:
        ds_dict = {key: value for key, value in ds.items()}
        ds_dict['rewards'] = (ds['rewards'] != 0.0) * -1.0
        ds = Dataset.create(**ds_dict)

    return ds


if EXECUTION_MODE == 'notebook':
    env, eval_env, train_dataset, val_dataset, dataset_paths = load_env_and_datasets_for_notebook(cfg)
    train_dataset = process_train_dataset(train_dataset, cfg)
    example_batch = train_dataset.sample(())

    dataset_rows = [
        ('Environment', cfg.env_name),
        ('Train transitions', f'{train_dataset.size:,}'),
        ('Observation shape', example_batch['observations'].shape),
        ('Action shape', example_batch['actions'].shape),
        ('Dataset proportion', cfg.dataset_proportion),
        ('Custom dataset files', len(dataset_paths)),
        ('Dataset replace interval', cfg.dataset_replace_interval),
        ('Sparse reward', cfg.sparse),
    ]
    display_kv('Dataset and Environment Summary', dataset_rows)
else:
    env = eval_env = train_dataset = val_dataset = example_batch = None
    dataset_paths = []
    display(Markdown('Dataset loading is skipped because EXECUTION_MODE is not notebook.'))


## 5. Agent Initialization

The TRQAM agent contains a critic ensemble, a slow flow actor, a fast flow actor, and target networks. When `pretrained_actor_path` is provided, the BC-trained flow policy is copied into the actor parameters used by the fine-tuning agent.


In [ ]:
def load_pretrained_actor(agent, checkpoint_path: str, config):
    checkpoint_path = str(Path(checkpoint_path).expanduser())
    with open(checkpoint_path, 'rb') as f:
        pretrained_dict = pickle.load(f)

    pretrained_params = pretrained_dict['agent']['network']['params']
    current_params = agent.network.params
    new_params = dict(current_params)

    pretrained_flow = None
    for key in ['modules_actor_slow', 'modules_actor_bc_flow', 'modules_actor_flow', 'modules_actor']:
        if key in pretrained_params:
            pretrained_flow = pretrained_params[key]
            display(Markdown(f'- Found pretrained flow policy: `{key}`'))
            break

    if pretrained_flow is None:
        display(Markdown('- Warning: no flow policy found in checkpoint.'))
        return agent

    loaded = False
    if 'modules_actor_slow' in current_params:
        new_params['modules_actor_slow'] = pretrained_flow
        new_params['modules_target_actor_slow'] = pretrained_flow
        if 'modules_actor_fast' in current_params:
            new_params['modules_actor_fast'] = pretrained_flow
            new_params['modules_target_actor_fast'] = pretrained_flow
        loaded = True

    if 'modules_actor_bc_flow' in current_params:
        new_params['modules_actor_bc_flow'] = pretrained_flow
        if 'modules_target_actor_bc_flow' in current_params:
            new_params['modules_target_actor_bc_flow'] = pretrained_flow
        loaded = True

    if 'modules_actor' in current_params and config['agent_name'] in ['cgql', 'ifql']:
        new_params['modules_actor'] = pretrained_flow
        if 'modules_target_actor' in current_params:
            new_params['modules_target_actor'] = pretrained_flow
        loaded = True

    if loaded:
        display(Markdown(f'- Pretrained actor loaded: `{checkpoint_path}`'))
        return agent.replace(network=agent.network.replace(params=new_params))

    display(Markdown('- Warning: checkpoint could not be mapped to current agent.'))
    return agent


random.seed(cfg.seed)
np.random.seed(cfg.seed)

if EXECUTION_MODE == 'notebook':
    agent_class = agents[agent_config['agent_name']]
    agent = agent_class.create(
        cfg.seed,
        example_batch['observations'],
        example_batch['actions'],
        agent_config,
    )

    if cfg.pretrained_actor_path is not None:
        agent = load_pretrained_actor(agent, cfg.pretrained_actor_path, agent_config)

    params_without_targets = {key: value for key, value in agent.network.params.items() if 'target' not in key}
    param_count = sum(leaf.size for leaf in jax.tree_util.tree_leaves(params_without_targets))
    agent_rows = [
        ('Agent', agent_config['agent_name']),
        ('Parameters without target nets', f'{param_count:,}'),
        ('Pretrained actor', cfg.pretrained_actor_path or 'not used'),
        ('Action chunking', cfg.action_chunking),
        ('Horizon length', cfg.horizon_length),
    ]
    display_kv('Agent Initialization Result', agent_rows)
else:
    agent = None
    display(Markdown('Agent initialization is skipped because EXECUTION_MODE is not notebook.'))


## 6. Training and Evaluation Functions

The following functions split the original `main.py` offline and online loops into notebook-executable units. Logs are written to CSV files as they are produced, so long full runs preserve intermediate evaluation results even before the final report cell executes.


In [ ]:
def checkpoint_step_from_path(path):
    match = re.search(r'params_(\d+)\.pkl$', Path(path).name)
    return int(match.group(1)) if match else -1


def normalized_config_value(value):
    if isinstance(value, list):
        return tuple(value)
    if isinstance(value, tuple):
        return tuple(value)
    return value


def saved_config_matches_cfg(run_dir, cfg):
    config_path = Path(run_dir) / 'notebook_config.json'
    if not config_path.exists():
        return True
    try:
        saved = json.loads(config_path.read_text())
    except Exception:
        return False

    keys = [
        'run_group', 'env_name', 'agent_name', 'seed', 'offline_steps', 'online_steps',
        'bc_only', 'horizon_length', 'action_chunking', 'batch_size', 'actor_hidden_dims',
        'value_hidden_dims', 'actor_layer_norm', 'flow_steps', 'kl_budget', 'inv_temp',
        'fql_alpha', 'edit_scale',
    ]
    current = asdict(cfg)
    for key in keys:
        if key in saved and normalized_config_value(saved[key]) != normalized_config_value(current.get(key)):
            return False
    return True


def completed_checkpoint_path(run_dir, cfg):
    run_dir = Path(run_dir)
    exact_path = run_dir / f'params_{cfg.offline_steps}.pkl'
    if exact_path.exists():
        return exact_path
    ckpts = sorted(run_dir.glob('params_*.pkl'), key=checkpoint_step_from_path)
    if ckpts and checkpoint_step_from_path(ckpts[-1]) >= cfg.offline_steps:
        return ckpts[-1]
    return None


def latest_offline_log_step(run_dir):
    path = Path(run_dir) / 'offline_agent.csv'
    if not path.exists():
        return None
    try:
        df = pd.read_csv(path, usecols=['step'])
    except Exception:
        return None
    if df.empty:
        return None
    steps = pd.to_numeric(df['step'], errors='coerce').dropna()
    return int(steps.max()) if not steps.empty else None


def is_completed_run_dir(run_dir, cfg):
    if cfg.offline_steps <= 0 or not saved_config_matches_cfg(run_dir, cfg):
        return False
    if completed_checkpoint_path(run_dir, cfg) is not None:
        return True
    last_step = latest_offline_log_step(run_dir)
    return last_step is not None and last_step >= cfg.offline_steps


def find_completed_run_dir(cfg):
    if not SKIP_COMPLETED_RUNS or cfg.offline_steps <= 0:
        return None
    base = Path(cfg.save_dir) / cfg.run_group / cfg.env_name
    runs = [path for path in base.glob('*') if path.is_dir()]
    completed_runs = [path for path in runs if is_completed_run_dir(path, cfg)]
    if not completed_runs:
        return None
    return max(completed_runs, key=lambda path: path.stat().st_mtime)


def make_save_dir(cfg, config):
    completed_run_dir = find_completed_run_dir(cfg)
    if completed_run_dir is not None:
        display(Markdown(f'Reusing completed run: `{completed_run_dir}`'))
        return completed_run_dir

    env_short = cfg.env_name.split('-')[0]
    timestamp = time.strftime('%Y%m%d-%H%M%S')
    exp_name = f'{cfg.tags}_{config["agent_name"]}_{env_short}_s{cfg.seed}_{timestamp}'
    save_dir = Path(cfg.save_dir) / cfg.run_group / cfg.env_name / exp_name
    save_dir.mkdir(parents=True, exist_ok=True)
    with open(save_dir / 'notebook_config.json', 'w') as f:
        json.dump(asdict(cfg), f, indent=2)
    return save_dir


def scalarize(row):
    out = {}
    for key, value in row.items():
        arr = np.asarray(value)
        out[key] = float(arr) if arr.shape == () else value
    return out


class NotebookLogger:
    def __init__(self, save_dir: Path):
        self.save_dir = save_dir
        self.rows = defaultdict(list)

    def log(self, data, prefix: str, step: int):
        row = scalarize(data)
        row['step'] = step
        self.rows[prefix].append(row)
        path = self.save_dir / f'{prefix}.csv'
        pd.DataFrame([row]).to_csv(path, mode='a', header=not path.exists(), index=False)

    def frame(self, prefix: str):
        return pd.DataFrame(self.rows[prefix])

    def save_all(self):
        for prefix in self.rows:
            self.frame(prefix).to_csv(self.save_dir / f'{prefix}.csv', index=False)


def update_success_rate_plot(save_dir, eval_df, cfg):
    if eval_df.empty or 'step' not in eval_df.columns or 'success' not in eval_df.columns:
        return None

    sr_df = eval_df[['step', 'success']].copy()
    sr_df['step'] = pd.to_numeric(sr_df['step'], errors='coerce')
    sr_df['success'] = pd.to_numeric(sr_df['success'], errors='coerce')
    sr_df = sr_df.dropna(subset=['step', 'success']).sort_values('step')
    if sr_df.empty:
        return None

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(sr_df['step'], sr_df['success'], marker='o', linewidth=2.0)
    ax.set_title('Success Rate by Step')
    ax.set_xlabel('step')
    ax.set_ylabel('SR')
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    fig_dir = Path(save_dir) / 'figures'
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig_path = fig_dir / 'success_rate_by_step.png'
    fig.savefig(fig_path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    return fig_path


def save_agent_snapshot(agent, save_dir, step, keep_last_only=False):
    save_path = Path(save_agent(agent, str(save_dir), step))
    if keep_last_only:
        for checkpoint_path in Path(save_dir).glob('params_*.pkl'):
            if checkpoint_path.resolve() != save_path.resolve():
                checkpoint_path.unlink()
    return save_path


ASYNC_EVAL_PROCESSES = []

ASYNC_EVAL_WORKER_CODE = r'''
import glob
import json
import os
import sys
import traceback
from pathlib import Path

meta_path = Path(sys.argv[1])
meta = json.loads(meta_path.read_text())
cuda_devices = meta.get('cuda_visible_devices')
if cuda_devices:
    if str(cuda_devices).lower() == 'cpu':
        os.environ['JAX_PLATFORMS'] = 'cpu'
        os.environ.pop('CUDA_VISIBLE_DEVICES', None)
    else:
        os.environ['CUDA_VISIBLE_DEVICES'] = str(cuda_devices)
os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

root = Path(meta['root'])
os.chdir(root)
sys.path.insert(0, str(root))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from agents import agents
from envs.env_utils import make_env_and_datasets
from envs.ogbench_utils import make_ogbench_env_and_datasets
from evaluation import evaluate
from utils.datasets import Dataset
from utils.flax_utils import restore_agent_with_file


def is_robomimic_env(env_name):
    if 'low_dim' not in env_name:
        return False
    task, dataset_type, _ = env_name.split('-')
    return task in ('lift', 'can', 'square', 'transport', 'tool_hang') and dataset_type in ('mh', 'ph')


def scalarize(row):
    out = {}
    for key, value in row.items():
        arr = np.asarray(value)
        out[key] = float(arr) if arr.shape == () else value
    return out


def process_train_dataset(ds, cfg):
    ds = Dataset.create(**ds)
    if cfg['dataset_proportion'] < 1.0 and cfg.get('ogbench_dataset_dir') is None:
        new_size = int(len(ds['masks']) * cfg['dataset_proportion'])
        ds = Dataset.create(**{key: value[:new_size] for key, value in ds.items()})
    if is_robomimic_env(cfg['env_name']):
        ds_dict = {key: value for key, value in ds.items()}
        ds_dict['rewards'] = ds['rewards'] - 1.0
        ds = Dataset.create(**ds_dict)
    if cfg.get('sparse'):
        ds_dict = {key: value for key, value in ds.items()}
        ds_dict['rewards'] = (ds['rewards'] != 0.0) * -1.0
        ds = Dataset.create(**ds_dict)
    return ds


def load_env_and_datasets(cfg):
    dataset_paths = []
    if cfg.get('ogbench_dataset_dir') is not None:
        dataset_paths = [
            path for path in sorted(glob.glob(str(Path(cfg['ogbench_dataset_dir']).expanduser() / '*.npz')))
            if '-val.npz' not in path
        ]
        if cfg['dataset_proportion'] < 1.0:
            subset_size = max(1, int(len(dataset_paths) * cfg['dataset_proportion']))
            dataset_paths = dataset_paths[:subset_size]
        if not dataset_paths:
            raise FileNotFoundError(f"No training npz files found in {cfg['ogbench_dataset_dir']}")
        return make_ogbench_env_and_datasets(cfg['env_name'], dataset_path=dataset_paths[0], compact_dataset=False)
    return make_env_and_datasets(cfg['env_name'])


def update_success_rate_plot(save_dir):
    eval_path = Path(save_dir) / 'eval.csv'
    if not eval_path.exists():
        return
    eval_df = pd.read_csv(eval_path)
    if eval_df.empty or 'step' not in eval_df.columns or 'success' not in eval_df.columns:
        return
    sr_df = eval_df[['step', 'success']].copy()
    sr_df['step'] = pd.to_numeric(sr_df['step'], errors='coerce')
    sr_df['success'] = pd.to_numeric(sr_df['success'], errors='coerce')
    sr_df = sr_df.dropna(subset=['step', 'success']).sort_values('step')
    if sr_df.empty:
        return
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(sr_df['step'], sr_df['success'], marker='o', linewidth=2.0)
    ax.set_title('Success Rate by Step')
    ax.set_xlabel('step')
    ax.set_ylabel('SR')
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig_dir = Path(save_dir) / 'figures'
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(fig_dir / 'success_rate_by_step.png', dpi=160, bbox_inches='tight')
    plt.close(fig)


status = {'step': int(meta['step']), 'ok': False}
exit_code = 0
try:
    cfg = meta['cfg']
    agent_config = meta['agent_config']
    for key in ['actor_hidden_dims', 'value_hidden_dims']:
        if key in agent_config:
            agent_config[key] = tuple(agent_config[key])

    env, eval_env, train_dataset, _ = load_env_and_datasets(cfg)
    train_dataset = process_train_dataset(train_dataset, cfg)
    example_batch = train_dataset.sample(())
    agent_class = agents[agent_config['agent_name']]
    agent = agent_class.create(cfg['seed'], example_batch['observations'], example_batch['actions'], agent_config)
    agent = restore_agent_with_file(agent, meta['checkpoint_path'])

    eval_info, _, _ = evaluate(
        agent=agent,
        env=eval_env,
        action_dim=example_batch['actions'].shape[-1],
        num_eval_episodes=cfg['eval_episodes'],
        num_video_episodes=cfg['video_episodes'],
        video_frame_skip=cfg['video_frame_skip'],
    )
    row = scalarize(eval_info)
    row['step'] = int(meta['step'])
    row['async_eval'] = 1.0
    save_dir = Path(meta['save_dir'])
    eval_path = save_dir / 'eval.csv'
    pd.DataFrame([row]).to_csv(eval_path, mode='a', header=not eval_path.exists(), index=False)
    update_success_rate_plot(save_dir)
    status = {'step': int(meta['step']), 'ok': True, 'eval_path': str(eval_path)}
except Exception as exc:
    exit_code = 1
    status = {'step': int(meta['step']), 'ok': False, 'error': repr(exc), 'traceback': traceback.format_exc()}
finally:
    Path(meta['status_path']).write_text(json.dumps(status, indent=2))
    for cleanup_path in [meta.get('eval_checkpoint_path'), str(meta_path)]:
        if cleanup_path:
            try:
                Path(cleanup_path).unlink(missing_ok=True)
            except Exception:
                pass
    sys.exit(exit_code)
'''


def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def reap_async_eval_processes():
    global ASYNC_EVAL_PROCESSES
    alive = []
    for item in ASYNC_EVAL_PROCESSES:
        retcode = item['process'].poll()
        if retcode is None:
            alive.append(item)
            continue
        item['log_handle'].close()
        if retcode != 0:
            display(Markdown(f"- Async eval failed at step `{item['step']}`. Log: `{item['log_path']}`"))
    ASYNC_EVAL_PROCESSES = alive
    return alive


def link_checkpoint_for_async_eval(checkpoint_path, save_dir, step):
    eval_ckpt_dir = Path(save_dir) / 'async_eval_checkpoints'
    eval_ckpt_dir.mkdir(parents=True, exist_ok=True)
    eval_checkpoint_path = eval_ckpt_dir / f'params_{step}.pkl'
    if eval_checkpoint_path.exists():
        eval_checkpoint_path.unlink()
    try:
        os.link(checkpoint_path, eval_checkpoint_path)
    except OSError:
        shutil.copy2(checkpoint_path, eval_checkpoint_path)
    return eval_checkpoint_path


def log_async_eval_skip(save_dir, step, reason):
    path = Path(save_dir) / 'eval_skipped.csv'
    row = {'step': int(step), 'reason': reason, 'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}
    pd.DataFrame([row]).to_csv(path, mode='a', header=not path.exists(), index=False)


def wait_for_async_eval_slot(step, checkpoint_path):
    reap_async_eval_processes()
    if not ASYNC_EVAL_PROCESSES:
        return

    display(Markdown(
        f'Async eval is busy; keeping checkpoint `{checkpoint_path}` and waiting before launching eval for step `{step}`.'
    ))
    while ASYNC_EVAL_PROCESSES:
        time.sleep(5)
        reap_async_eval_processes()


def launch_async_eval(checkpoint_path, step, cfg, agent_config, save_dir):
    reap_async_eval_processes()
    eval_checkpoint_path = link_checkpoint_for_async_eval(Path(checkpoint_path), save_dir, step)
    wait_for_async_eval_slot(step, eval_checkpoint_path)
    request_dir = Path(save_dir) / 'async_eval_requests'
    status_dir = Path(save_dir) / 'async_eval_status'
    log_dir = Path(save_dir) / 'async_eval_logs'
    for path in [request_dir, status_dir, log_dir]:
        path.mkdir(parents=True, exist_ok=True)

    meta_path = request_dir / f'eval_{step}.json'
    status_path = status_dir / f'eval_{step}.json'
    log_path = log_dir / f'eval_{step}.log'
    meta = {
        'root': str(ROOT),
        'save_dir': str(save_dir),
        'step': int(step),
        'cfg': asdict(cfg),
        'agent_config': dict(agent_config),
        'checkpoint_path': str(eval_checkpoint_path),
        'eval_checkpoint_path': str(eval_checkpoint_path),
        'status_path': str(status_path),
        'cuda_visible_devices': cfg.async_eval_cuda_visible_devices,
    }
    meta_path.write_text(json.dumps(json_safe(meta), indent=2))

    env_vars = os.environ.copy()
    env_vars['WANDB_MODE'] = 'disabled'
    env_vars['MUJOCO_GL'] = 'egl'
    env_vars['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
    if cfg.async_eval_cuda_visible_devices:
        if str(cfg.async_eval_cuda_visible_devices).lower() == 'cpu':
            env_vars['JAX_PLATFORMS'] = 'cpu'
            env_vars.pop('CUDA_VISIBLE_DEVICES', None)
        else:
            env_vars['CUDA_VISIBLE_DEVICES'] = str(cfg.async_eval_cuda_visible_devices)

    log_handle = open(log_path, 'w')
    process = subprocess.Popen(
        [sys.executable, '-c', ASYNC_EVAL_WORKER_CODE, str(meta_path)],
        cwd=str(ROOT),
        env=env_vars,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )
    ASYNC_EVAL_PROCESSES.append(
        {'step': int(step), 'process': process, 'log_path': log_path, 'log_handle': log_handle}
    )
    return True


if EXECUTION_MODE == 'notebook':
    save_dir = make_save_dir(cfg, agent_config)
else:
    save_dir = Path(cfg.save_dir) / 'controller_outputs' / RUN_PROFILE
    save_dir.mkdir(parents=True, exist_ok=True)

logger = NotebookLogger(save_dir)
report_figures = {}
ACTIVE_SETUP_PROFILE = RUN_PROFILE

display_kv('Output Path', [('Current notebook output directory', f'`{save_dir}`')])


In [ ]:
def maybe_replace_ogbench_dataset(step, cfg, dataset_paths, dataset_idx, env):
    if not dataset_paths or cfg.dataset_replace_interval == 0:
        return None, dataset_idx
    if step % cfg.dataset_replace_interval != 0:
        return None, dataset_idx

    dataset_idx = (dataset_idx + 1) % len(dataset_paths)
    train_dataset, _ = make_ogbench_env_and_datasets(
        cfg.env_name,
        dataset_path=dataset_paths[dataset_idx],
        compact_dataset=False,
        dataset_only=True,
        cur_env=env,
    )
    return process_train_dataset(train_dataset, cfg), dataset_idx


def run_offline_phase(agent, env, eval_env, train_dataset, example_batch, cfg, config, logger, save_dir, dataset_paths):
    if cfg.offline_steps <= 0:
        return agent, train_dataset

    completed_run_dir = find_completed_run_dir(cfg)
    if completed_run_dir is not None and Path(save_dir).resolve() == Path(completed_run_dir).resolve():
        final_checkpoint = completed_checkpoint_path(completed_run_dir, cfg)
        if final_checkpoint is not None:
            try:
                agent = restore_agent_with_file(agent, str(final_checkpoint))
                restore_message = f'restored `{final_checkpoint.name}`'
            except Exception as exc:
                restore_message = f'checkpoint restore skipped: {type(exc).__name__}: {exc}'
        else:
            restore_message = 'no final checkpoint found, using saved logs only'

        eval_path = Path(completed_run_dir) / 'eval.csv'
        if eval_path.exists():
            update_success_rate_plot(completed_run_dir, pd.read_csv(eval_path), cfg)

        display_kv(
            'Training Skipped',
            [
                ('Reason', f'completed run already has {cfg.offline_steps:,} offline steps'),
                ('Run directory', f'`{completed_run_dir}`'),
                ('Checkpoint', restore_message),
            ],
        )
        return agent, train_dataset

    dataset_idx = 0
    action_dim = example_batch['actions'].shape[-1]
    last_saved_step = None

    for step in trange(1, cfg.offline_steps + 1, desc='offline', mininterval=30, smoothing=0.0):
        new_dataset, dataset_idx = maybe_replace_ogbench_dataset(step, cfg, dataset_paths, dataset_idx, env)
        if new_dataset is not None:
            train_dataset = new_dataset

        batch = train_dataset.sample_sequence(
            config['batch_size'],
            sequence_length=cfg.horizon_length,
            discount=config['discount'],
        )
        agent, info = agent.update(batch)

        if step == 1 or step % cfg.log_interval == 0:
            logger.log(info, 'offline_agent', step=step)

        should_eval = step == cfg.offline_steps or (cfg.eval_interval > 0 and step % cfg.eval_interval == 0)
        should_save = cfg.save_interval > 0 and step % cfg.save_interval == 0
        checkpoint_path = None

        if should_save:
            checkpoint_path = save_agent_snapshot(agent, save_dir, step, cfg.keep_last_checkpoint_only)
            last_saved_step = step

        if should_eval and cfg.async_eval:
            if checkpoint_path is None:
                checkpoint_path = save_agent_snapshot(agent, save_dir, step, cfg.keep_last_checkpoint_only)
                last_saved_step = step
            launch_async_eval(checkpoint_path, step, cfg, config, save_dir)
        elif should_eval:
            eval_info, _, _ = evaluate(
                agent=agent,
                env=eval_env,
                action_dim=action_dim,
                num_eval_episodes=cfg.eval_episodes,
                num_video_episodes=cfg.video_episodes,
                video_frame_skip=cfg.video_frame_skip,
            )
            logger.log(eval_info, 'eval', step=step)
            update_success_rate_plot(save_dir, logger.frame('eval'), cfg)
        elif cfg.async_eval and step % cfg.log_interval == 0:
            reap_async_eval_processes()

    if cfg.save_interval > 0 and last_saved_step != cfg.offline_steps:
        save_agent_snapshot(agent, save_dir, cfg.offline_steps, cfg.keep_last_checkpoint_only)

    if cfg.async_eval:
        reap_async_eval_processes()

    logger.save_all()
    return agent, train_dataset

In [ ]:
def run_online_phase(agent, env, eval_env, train_dataset, example_batch, cfg, config, logger, save_dir):
    if cfg.online_steps <= 0:
        return agent

    if cfg.kl_budget_online is not None:
        config['kl_budget'] = cfg.kl_budget_online
        new_config = dict(agent.config)
        new_config['kl_budget'] = cfg.kl_budget_online
        object.__setattr__(agent, 'config', flax.core.FrozenDict(new_config))

    action_dim = example_batch['actions'].shape[-1]
    if cfg.balanced_sampling:
        replay_buffer = ReplayBuffer.create(example_batch, size=cfg.online_steps)
    else:
        replay_buffer = ReplayBuffer.create_from_initial_dataset(
            dict(train_dataset),
            size=train_dataset.size + cfg.online_steps,
        )

    online_rng = jax.random.PRNGKey(cfg.seed + 1)
    observation, _ = env.reset()
    action_queue = []
    latest_update_info = None

    for step in trange(1, cfg.online_steps + 1, desc='online', mininterval=30, smoothing=0.0):
        log_step = cfg.offline_steps + step
        online_rng, key = jax.random.split(online_rng)

        if not action_queue:
            if cfg.balanced_sampling and step < cfg.start_training:
                action = np.random.rand(action_dim) * 2.0 - 1.0
            else:
                action = agent.sample_actions(observations=observation, rng=key)
            for chunk_action in np.asarray(action).reshape(-1, action_dim):
                action_queue.append(chunk_action)

        action = np.clip(action_queue.pop(0), -1.0, 1.0)
        next_observation, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        env_info = {key: value for key, value in info.items() if key.startswith('distance')}
        if env_info:
            logger.log(env_info, 'env', step=log_step)

        if is_robomimic_env(cfg.env_name):
            reward = reward - 1.0
        if cfg.sparse:
            reward = (reward != 0.0) * -1.0

        replay_buffer.add_transition(
            dict(
                observations=observation,
                actions=action,
                rewards=reward,
                terminals=float(done),
                masks=1.0 - float(terminated),
                next_observations=next_observation,
            )
        )

        if done:
            observation, _ = env.reset()
            action_queue = []
        else:
            observation = next_observation

        if step >= cfg.start_training:
            if cfg.balanced_sampling:
                half_batch = config['batch_size'] // 2
                dataset_batch = train_dataset.sample_sequence(
                    half_batch * cfg.utd_ratio,
                    sequence_length=cfg.horizon_length,
                    discount=config['discount'],
                )
                replay_batch = replay_buffer.sample_sequence(
                    half_batch * cfg.utd_ratio,
                    sequence_length=cfg.horizon_length,
                    discount=config['discount'],
                )
                batch = {
                    key: np.concatenate(
                        [
                            dataset_batch[key].reshape((cfg.utd_ratio, half_batch) + dataset_batch[key].shape[1:]),
                            replay_batch[key].reshape((cfg.utd_ratio, half_batch) + replay_batch[key].shape[1:]),
                        ],
                        axis=1,
                    )
                    for key in dataset_batch
                }
            else:
                batch = replay_buffer.sample_sequence(
                    config['batch_size'] * cfg.utd_ratio,
                    sequence_length=cfg.horizon_length,
                    discount=config['discount'],
                )
                batch = jax.tree_util.tree_map(
                    lambda x: x.reshape((cfg.utd_ratio, config['batch_size']) + x.shape[1:]),
                    batch,
                )
            agent, latest_update_info = agent.batch_update(batch)

        if latest_update_info is not None and (step == cfg.start_training or step % cfg.log_interval == 0):
            logger.log(latest_update_info, 'online_agent', step=log_step)

        should_eval = step == cfg.online_steps or (cfg.eval_interval > 0 and step % cfg.eval_interval == 0)
        if should_eval:
            eval_info, _, _ = evaluate(
                agent=agent,
                env=eval_env,
                action_dim=action_dim,
                num_eval_episodes=cfg.eval_episodes,
                num_video_episodes=cfg.video_episodes,
                video_frame_skip=cfg.video_frame_skip,
            )
            logger.log(eval_info, 'eval', step=log_step)
            update_success_rate_plot(save_dir, logger.frame('eval'), cfg)

        if cfg.save_interval > 0 and step % cfg.save_interval == 0:
            save_agent_snapshot(agent, save_dir, log_step, cfg.keep_last_checkpoint_only)

    logger.save_all()
    return agent

## 7. BC and Offline RL Execution

Run exactly one stage cell below. Each stage cell calls `ensure_profile_setup(...)` first, so running 7B after 7A automatically switches the kernel state from BC setup to TRQAM offline-RL setup and loads the latest BC checkpoint.

BC writes one final checkpoint. TRQAM/QAM offline RL saves the latest training snapshot every 1K steps and launches async eval in a separate subprocess. Eval writes `eval.csv`, refreshes the SR plot, and deletes only its temporary eval checkpoint after that eval finishes. If the eval worker is still busy at a later 1K step, the notebook keeps that step's eval checkpoint and waits instead of skipping it.


In [ ]:
def summarize_stage(stage_name):
    logger.save_all()
    log_files = sorted(path.name for path in save_dir.glob('*.csv'))
    display_kv(
        f'{stage_name} Completed',
        [
            ('Saved logs', ', '.join(log_files) if log_files else 'no csv logs'),
            ('Save directory', f'`{save_dir}`'),
        ],
    )


def require_profile(expected):
    if EXECUTION_MODE != 'notebook':
        raise RuntimeError('This notebook is configured for notebook-local execution only.')
    if RUN_PROFILE != expected:
        raise RuntimeError(
            f'This cell requires RUN_PROFILE={expected!r}, but current RUN_PROFILE={RUN_PROFILE!r}.\n'
            'Edit RUN_PROFILE in Section 3, rerun the setup cells from Section 3 through Section 6, '
            'then run this stage cell again.'
        )


def close_existing_envs():
    for name in ['env', 'eval_env']:
        old_env = globals().get(name)
        if old_env is not None and hasattr(old_env, 'close'):
            try:
                old_env.close()
            except Exception:
                pass


def ensure_profile_setup(profile):
    global RUN_PROFILE, cfg, agent_config
    global env, eval_env, train_dataset, val_dataset, dataset_paths, example_batch
    global agent, save_dir, logger, report_figures, ACTIVE_SETUP_PROFILE

    if EXECUTION_MODE != 'notebook':
        raise RuntimeError('This notebook is configured for notebook-local execution only.')

    required_names = ['cfg', 'agent_config', 'env', 'eval_env', 'train_dataset', 'example_batch', 'agent', 'save_dir', 'logger']
    if globals().get('ACTIVE_SETUP_PROFILE') == profile and all(name in globals() for name in required_names):
        return

    close_existing_envs()
    RUN_PROFILE = profile
    cfg = configure_profile(profile)
    agent_config = build_agent_config(cfg)

    env, eval_env, train_dataset, val_dataset, dataset_paths = load_env_and_datasets_for_notebook(cfg)
    train_dataset = process_train_dataset(train_dataset, cfg)
    example_batch = train_dataset.sample(())

    random.seed(cfg.seed)
    np.random.seed(cfg.seed)
    agent_class = agents[agent_config['agent_name']]
    agent = agent_class.create(cfg.seed, example_batch['observations'], example_batch['actions'], agent_config)
    if cfg.pretrained_actor_path is not None:
        agent = load_pretrained_actor(agent, cfg.pretrained_actor_path, agent_config)

    save_dir = make_save_dir(cfg, agent_config)
    logger = NotebookLogger(save_dir)
    report_figures = {}
    ACTIVE_SETUP_PROFILE = profile

    display_kv(
        f'Profile Setup: {profile}',
        [
            ('Agent', agent_config['agent_name']),
            ('Environment', cfg.env_name),
            ('Pretrained actor', cfg.pretrained_actor_path or 'not used'),
            ('Output directory', f'`{save_dir}`'),
            ('Eval interval', cfg.eval_interval),
            ('Save interval', cfg.save_interval),
            ('Keep last checkpoint only', cfg.keep_last_checkpoint_only),
            ('Async eval', cfg.async_eval),
            ('Async eval busy policy', 'wait and keep eval checkpoint'),
            ('Async eval CUDA_VISIBLE_DEVICES', cfg.async_eval_cuda_visible_devices or 'inherit'),
        ],
    )



### 7A. BC Pretraining

Run this cell only when `RUN_PROFILE='full_bc'`. It trains the BC flow policy for 300K steps.


In [ ]:
ensure_profile_setup('full_bc')

agent, train_dataset = run_offline_phase(
    agent=agent,
    env=env,
    eval_env=eval_env,
    train_dataset=train_dataset,
    example_batch=example_batch,
    cfg=cfg,
    config=agent_config,
    logger=logger,
    save_dir=save_dir,
    dataset_paths=dataset_paths,
)

summarize_stage('BC Pretraining')



### 7B. TRQAM Offline RL

Run this cell to load the latest BC checkpoint and start 1M-step TRQAM offline RL. The cell switches setup to `RUN_PROFILE='full_offline_trqam'` automatically.


In [ ]:
ensure_profile_setup('full_offline_trqam')

agent, train_dataset = run_offline_phase(
    agent=agent,
    env=env,
    eval_env=eval_env,
    train_dataset=train_dataset,
    example_batch=example_batch,
    cfg=cfg,
    config=agent_config,
    logger=logger,
    save_dir=save_dir,
    dataset_paths=dataset_paths,
)

summarize_stage('TRQAM Offline RL')



### 7C. QAM Offline RL

Run this cell to load the latest BC checkpoint and start 1M-step QAM offline RL. The cell switches setup to `RUN_PROFILE='full_offline_qam'` automatically.


In [ ]:
ensure_profile_setup('full_offline_qam')

agent, train_dataset = run_offline_phase(
    agent=agent,
    env=env,
    eval_env=eval_env,
    train_dataset=train_dataset,
    example_batch=example_batch,
    cfg=cfg,
    config=agent_config,
    logger=logger,
    save_dir=save_dir,
    dataset_paths=dataset_paths,
)

summarize_stage('QAM Offline RL')



### 7D. Cube-Triple BC Pretraining

Run this cell to start 300K-step cube-triple BC pretraining. It uses `cube-triple-play-singletask-task2-v0`, action chunk horizon 1, width 1024, and layer norm. If `TRQAM_CUBE_TRIPLE_DATASET_DIR` or `~/.ogbench/data/cube-triple-play-10m-v0` exists, it cycles those npz files; otherwise it falls back to the cached OGBench dataset.

In [ ]:
ensure_profile_setup('cube_triple_bc')

agent, train_dataset = run_offline_phase(
    agent=agent,
    env=env,
    eval_env=eval_env,
    train_dataset=train_dataset,
    example_batch=example_batch,
    cfg=cfg,
    config=agent_config,
    logger=logger,
    save_dir=save_dir,
    dataset_paths=dataset_paths,
)

summarize_stage('Cube-Triple BC Pretraining')



### 7E. Cube-Triple TRQAM Offline RL

Run this cell to load the latest cube-triple BC checkpoint and start 1M-step TRQAM offline RL.

In [ ]:
ensure_profile_setup('cube_triple_offline_trqam')

agent, train_dataset = run_offline_phase(
    agent=agent,
    env=env,
    eval_env=eval_env,
    train_dataset=train_dataset,
    example_batch=example_batch,
    cfg=cfg,
    config=agent_config,
    logger=logger,
    save_dir=save_dir,
    dataset_paths=dataset_paths,
)

summarize_stage('Cube-Triple TRQAM Offline RL')



### 7F. Cube-Triple QAM Offline RL

Run this cell to load the latest cube-triple BC checkpoint and start 1M-step QAM offline RL.

In [ ]:
ensure_profile_setup('cube_triple_offline_qam')

agent, train_dataset = run_offline_phase(
    agent=agent,
    env=env,
    eval_env=eval_env,
    train_dataset=train_dataset,
    example_batch=example_batch,
    cfg=cfg,
    config=agent_config,
    logger=logger,
    save_dir=save_dir,
    dataset_paths=dataset_paths,
)

summarize_stage('Cube-Triple QAM Offline RL')



## 8. Live Run Status

Run this cell whenever the notebook output looks stale. It reads the CSV/checkpoint files from disk, so it works even when the Jupyter front-end is not replaying live output.

In [ ]:
def latest_run_dir(run_group, env_name):
    base = Path('exp_full') / run_group / env_name
    runs = [path for path in base.glob('*') if path.is_dir()]
    return max(runs, key=lambda path: path.stat().st_mtime) if runs else None


def latest_checkpoint_step(run_dir):
    def parse_step(path):
        match = re.search(r'params_(\d+)\.pkl$', path.name)
        return int(match.group(1)) if match else -1
    ckpts = sorted(Path(run_dir).glob('params_*.pkl'), key=parse_step)
    return parse_step(ckpts[-1]) if ckpts else None


def latest_csv_step(run_dir, filename):
    path = Path(run_dir) / filename
    if not path.exists():
        return None, None
    df = pd.read_csv(path)
    if df.empty or 'step' not in df.columns:
        return None, df
    return int(pd.to_numeric(df['step'], errors='coerce').dropna().max()), df


def current_async_eval_step(run_dir):
    requests = sorted((Path(run_dir) / 'async_eval_requests').glob('eval_*.json'))
    if not requests:
        return None
    match = re.search(r'eval_(\d+)\.json$', requests[-1].name)
    return int(match.group(1)) if match else requests[-1].name


def summarize_disk_run(label, run_group, env_name):
    run_dir = latest_run_dir(run_group, env_name)
    if run_dir is None:
        return {'run': label, 'status': 'no run'}
    train_step, train_df = latest_csv_step(run_dir, 'offline_agent.csv')
    eval_step, eval_df = latest_csv_step(run_dir, 'eval.csv')
    skipped_step, skipped_df = latest_csv_step(run_dir, 'eval_skipped.csv')
    latest_success = None
    if eval_df is not None and not eval_df.empty and 'success' in eval_df.columns:
        latest_success = pd.to_numeric(eval_df['success'], errors='coerce').dropna().iloc[-1]
    return {
        'run': label,
        'train_step': train_step,
        'eval_step': eval_step,
        'latest_success': latest_success,
        'latest_checkpoint_step': latest_checkpoint_step(run_dir),
        'active_async_eval_step': current_async_eval_step(run_dir),
        'last_skipped_eval_step': skipped_step,
        'updated': time.strftime('%Y-%m-%d %H:%M:%S'),
        'run_dir': str(run_dir),
    }


HUMANOID_MEDIUM_ENV = 'humanoidmaze-medium-navigate-singletask-task1-v0'
status_df = pd.DataFrame([
    summarize_disk_run('Humanoid BC', 'bc_pretrain_humanoid_medium', HUMANOID_MEDIUM_ENV),
    summarize_disk_run('Humanoid TRQAM', 'offline_rl_humanoid_medium', HUMANOID_MEDIUM_ENV),
    summarize_disk_run('Humanoid QAM', 'offline_qam_humanoid_medium', HUMANOID_MEDIUM_ENV),
    summarize_disk_run('Cube-Triple BC', 'bc_pretrain_cube_triple', CUBE_TRIPLE_ENV),
    summarize_disk_run('Cube-Triple TRQAM', 'offline_rl_cube_triple', CUBE_TRIPLE_ENV),
    summarize_disk_run('Cube-Triple QAM', 'offline_qam_cube_triple', CUBE_TRIPLE_ENV),
])
display(status_df)


## 9. Optional Legacy Checkpoint Re-Eval

The main SR curve now comes from `eval.csv`, written every 1K RL steps during training. This optional cell is only for legacy runs that still have multiple saved `params_*.pkl` checkpoints and need SR recomputed after training.


In [ ]:
# Optional legacy checkpoint eval without retraining. New RL runs already write eval.csv every 1K steps.
RUN_CHECKPOINT_REEVAL = False
CHECKPOINT_REEVAL_EPISODES = cfg.eval_episodes


def checkpoint_step(path):
    match = re.search(r'params_(\d+)\.pkl$', path.name)
    if not match:
        raise ValueError(f'Cannot parse checkpoint step from {path}')
    return int(match.group(1))


def find_latest_run_with_checkpoints(cfg):
    patterns = [
        Path(cfg.save_dir).glob(f'{cfg.run_group}/{cfg.env_name}/*/params_*.pkl'),
        Path(cfg.save_dir).glob(f'*/{cfg.run_group}/{cfg.env_name}/*/params_*.pkl'),
        Path(cfg.save_dir).glob(f'bc_pretrain_humanoid_medium/{cfg.env_name}/*/params_*.pkl'),
    ]
    ckpts = [path for pattern in patterns for path in pattern]
    if not ckpts:
        return None
    return max((path.parent for path in ckpts), key=lambda d: d.stat().st_mtime)


def recompute_checkpoint_eval(run_dir):
    checkpoint_paths = sorted(Path(run_dir).glob('params_*.pkl'), key=checkpoint_step)
    if not checkpoint_paths:
        raise FileNotFoundError(f'No params_*.pkl checkpoints found in {run_dir}')

    rows = []
    action_dim = example_batch['actions'].shape[-1]
    for checkpoint_path in checkpoint_paths:
        step = checkpoint_step(checkpoint_path)
        eval_agent = restore_agent_with_file(agent, str(checkpoint_path))
        eval_info, _, _ = evaluate(
            agent=eval_agent,
            env=eval_env,
            action_dim=action_dim,
            num_eval_episodes=CHECKPOINT_REEVAL_EPISODES,
            num_video_episodes=cfg.video_episodes,
            video_frame_skip=cfg.video_frame_skip,
        )
        row = scalarize(eval_info)
        row['step'] = step
        rows.append(row)

    recomputed_df = pd.DataFrame(rows).sort_values('step')
    out_path = Path(run_dir) / 'eval_recomputed.csv'
    recomputed_df.to_csv(out_path, index=False)
    display(Markdown(f'Recomputed checkpoint eval saved to: `{out_path}`'))
    display(recomputed_df[['step', 'success']] if 'success' in recomputed_df.columns else recomputed_df)
    return recomputed_df


if RUN_CHECKPOINT_REEVAL:
    reeval_run_dir = save_dir if any(Path(save_dir).glob('params_*.pkl')) else find_latest_run_with_checkpoints(cfg)
    if reeval_run_dir is None:
        raise FileNotFoundError('No checkpoint run directory was found for re-evaluation.')
    save_dir = Path(reeval_run_dir)
    eval_df = recompute_checkpoint_eval(save_dir)
else:
    display(Markdown('Checkpoint re-evaluation is disabled. New RL runs use `eval.csv` for the SR curve.'))



## 10. Result Tables

Training logs are saved as `offline_agent.csv`, `online_agent.csv`, and `eval.csv`. For RL profiles, `eval.csv` contains the 1K-step SR curve.


In [ ]:
if 'report_figures' not in globals():
    report_figures = {}


def load_log_frame(prefix):
    if 'logger' in globals():
        frame = logger.frame(prefix)
        if not frame.empty:
            return frame
    if 'save_dir' not in globals():
        raise RuntimeError(
            'No in-memory logger or saved CSV logs were found. Run the setup cells through '
            'the training cell first, then rerun this result section.'
        )
    csv_path = Path(save_dir) / f'{prefix}.csv'
    if csv_path.exists():
        return pd.read_csv(csv_path)
    if prefix == 'eval':
        recomputed_path = Path(save_dir) / 'eval_recomputed.csv'
        if recomputed_path.exists():
            return pd.read_csv(recomputed_path)
    return pd.DataFrame()


offline_df = load_log_frame('offline_agent')
online_df = load_log_frame('online_agent')
eval_df = load_log_frame('eval')

display(Markdown('### Final Evaluation Result'))
if not eval_df.empty:
    display(eval_df.tail(1))
else:
    display(pd.DataFrame({'message': ['no eval logs']}))

display(Markdown('### Recent Offline Training Logs'))
if not offline_df.empty:
    display(offline_df.tail())
else:
    display(pd.DataFrame({'message': ['no offline logs']}))

if not online_df.empty:
    display(Markdown('### Recent Online Training Logs'))
    display(online_df.tail())


In [ ]:
def slugify(text):
    slug = re.sub(r'[^A-Za-z0-9]+', '_', text).strip('_').lower()
    return slug or 'figure'


def plot_metrics(df, title, metric_candidates):
    metrics = [
        metric for metric in metric_candidates
        if metric in df.columns and pd.api.types.is_numeric_dtype(df[metric])
    ]
    if df.empty or not metrics:
        display(Markdown(f'_No metrics to plot for **{title}**._'))
        return None

    fig, ax = plt.subplots(figsize=(9, 4))
    df.plot(x='step', y=metrics, marker='o', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('step')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    fig_dir = save_dir / 'figures'
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig_path = fig_dir / f'{slugify(title)}.png'
    fig.savefig(fig_path, dpi=160, bbox_inches='tight')
    report_figures[title] = fig_path.relative_to(save_dir).as_posix()
    display(fig)
    plt.close(fig)
    return fig_path


def plot_success_rate(eval_df, cfg):
    if eval_df.empty or 'step' not in eval_df.columns or 'success' not in eval_df.columns:
        display(Markdown('_No success-rate evaluations to plot._'))
        return None

    sr_df = eval_df[['step', 'success']].copy()
    sr_df['step'] = pd.to_numeric(sr_df['step'], errors='coerce')
    sr_df['success'] = pd.to_numeric(sr_df['success'], errors='coerce')
    sr_df = sr_df.dropna(subset=['step', 'success'])
    if sr_df.empty:
        display(Markdown('_No success-rate evaluations to plot._'))
        return None

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(sr_df['step'], sr_df['success'], marker='o', linewidth=2.0)
    ax.set_title('Success Rate by Step')
    ax.set_xlabel('step')
    ax.set_ylabel('SR')
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    fig_dir = save_dir / 'figures'
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig_path = fig_dir / 'success_rate_by_step.png'
    fig.savefig(fig_path, dpi=160, bbox_inches='tight')
    report_figures['Success Rate by Step'] = fig_path.relative_to(save_dir).as_posix()
    display(fig)
    plt.close(fig)
    return fig_path


plot_success_rate(eval_df, cfg)


## 11. Automatically Generated Report

The final cell renders a report block from the current run and writes the same content to `report.html` inside the run directory. After a full run, this section contains the configuration, final evaluation metrics, recent training diagnostics, and saved plots.


In [ ]:
def _format_float(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    return value


def _df_to_report_html(df, empty_message='No records', max_rows=5):
    if df.empty:
        return f'<p class="trqam-muted">{html.escape(empty_message)}</p>'
    view = df.tail(max_rows).copy()
    return view.to_html(index=False, classes='report-table', border=0, escape=True, float_format=lambda x: f'{x:.6g}')


def _metric_card(label, value):
    return f'<div class="trqam-kpi"><div class="label">{html.escape(str(label))}</div><div class="value">{html.escape(str(value))}</div></div>'


def _last_value(df, key, default='n/a'):
    if df.empty or key not in df.columns:
        return default
    return _format_float(df.iloc[-1][key])


def build_report_html(cfg, save_dir, eval_df, offline_df, online_df, report_figures):
    final_success = _last_value(eval_df, 'success')
    final_return = _last_value(eval_df, 'episode.return')
    final_length = _last_value(eval_df, 'episode.length')
    final_flow_loss = _last_value(offline_df, 'actor/flow_loss')
    final_fast_loss = _last_value(offline_df, 'actor/fast_loss')
    final_path_kl = _last_value(offline_df, 'actor/path_kl')

    cards = ''.join([
        _metric_card('Run profile', RUN_PROFILE),
        _metric_card('Environment', cfg.env_name),
        _metric_card('Seed', cfg.seed),
        _metric_card('Offline steps', cfg.offline_steps),
        _metric_card('Online steps', cfg.online_steps),
        _metric_card('JAX backend', runtime_info.get('jax_backend', 'n/a')),
        _metric_card('SR', final_success),
        _metric_card('Return', final_return),
        _metric_card('Episode length', final_length),
        _metric_card('Flow loss', final_flow_loss),
        _metric_card('Fast loss', final_fast_loss),
        _metric_card('Path KL', final_path_kl),
    ])

    config_table = pd.DataFrame([(key, fmt_value(value)) for key, value in asdict(cfg).items()], columns=['Item', 'Value']).to_html(
        index=False,
        classes='report-table',
        border=0,
        escape=True,
    )

    figures_html = ''
    for title, rel_path in report_figures.items():
        figures_html += f'<h3>{html.escape(title)}</h3><img src="{html.escape(rel_path)}" alt="{html.escape(title)}" style="max-width:100%; border:1px solid #d0d7de; border-radius:6px;">'
    if not figures_html:
        figures_html = '<p class="trqam-muted">No figures were generated.</p>'

    return f'''
<div class="trqam-report">
  <h2>TRQAM Experiment Report</h2>
  <p class="trqam-muted">Generated at: {html.escape(time.strftime('%Y-%m-%d %H:%M:%S'))}</p>

  <div class="trqam-callout">
    This report was generated automatically from the current notebook run. For the full paper-style pipeline,
    run BC pretraining first, then run TRQAM fine-tuning from the saved BC checkpoint.
  </div>

  <h3>Key Metrics</h3>
  <div class="trqam-kpi-grid">{cards}</div>

  <h3>Method Summary</h3>
  <p>
    TRQAM starts from a flow policy and improves its vector field using an adjoint target derived from the Q-function action gradient.
    A path-space KL trust region and dual variable limit how far the fine-tuned policy can move from the pretrained base policy.
  </p>

  <h3>Experiment Configuration</h3>
  {config_table}

  <h3>Final Evaluation Log</h3>
  {_df_to_report_html(eval_df, 'No evaluation logs.', max_rows=1)}

  <h3>Recent Offline Training Logs</h3>
  {_df_to_report_html(offline_df, 'No offline training logs.', max_rows=5)}

  <h3>Recent Online Training Logs</h3>
  {_df_to_report_html(online_df, 'No online training logs.', max_rows=5)}

  <h3>Figures</h3>
  {figures_html}

  <h3>Interpretation Guide</h3>
  <p>
    `actor/flow_loss` tracks behavior-cloning quality, `actor/path_kl` and `dual/lambda` track trust-region usage,
    The main curve is SR (`success`) by offline step; `episode.return` and diagnostic losses are included only as supporting logs.
  </p>

  <p class="trqam-muted">Run directory: {html.escape(str(save_dir))}</p>
</div>
'''


def render_report_summary(cfg, save_dir, eval_df, offline_df, online_df):
    report_html = build_report_html(cfg, save_dir, eval_df, offline_df, online_df, report_figures)
    display(HTML(report_html))

    full_html = f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>TRQAM Report</title>
{REPORT_CSS}
</head>
<body style="max-width: 980px; margin: 32px auto; padding: 0 20px;">
{report_html}
</body>
</html>
'''
    report_path = save_dir / 'report.html'
    report_path.write_text(full_html, encoding='utf-8')
    display(Markdown(f'Report HTML saved to: `{report_path}`'))
    return report_path


report_path = render_report_summary(cfg, save_dir, eval_df, offline_df, online_df)
